# Porto Taxi ETA Exploration

Notebook nay mo dau cho bai toan du doan ETA tu GPS trajectory theo thoi gian thuc. Muc tieu cua chung ta la doc du lieu goc `train.csv`, hieu schema, va bien moi chuyen di thanh nhung dac trung co the dung de du doan thoi gian con lai den dich.

In [ ]:
import ast
from pathlib import Path

import pandas as pd

DATA_PATH = Path("../train.csv")
SAMPLE_ROWS = 50_000

df = pd.read_csv(DATA_PATH, nrows=SAMPLE_ROWS)
print(f"Loaded {len(df):,} rows from {DATA_PATH.resolve()}")
df.head(3)

## Kiem tra schema ban dau

Sau khi doc du lieu, viec dau tien la xac nhan kich thuoc, ten cot, va kieu du lieu. Buoc nay giup minh thay ngay cot nao la metadata cua chuyen di, cot nao chua thong tin GPS trajectory, va cot nao co the dung de tao feature cho ETA.

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nDtypes:")
display(df.dtypes)
display(df.head(3))

## Kiem tra chat luong du lieu

Truoc khi parse trajectory, can kiem tra nhanh do day du cua du lieu. Trong bo Porto Taxi, `MISSING_DATA` rat quan trong vi no bao hieu mot chuyen di bi thieu GPS point. Neu dung de train ETA, nhung dong nay thuong nen loc ra o giai doan tien xu ly.

In [3]:
missing_per_column = df.isna().sum().sort_values(ascending=False)
missing_data_counts = df["MISSING_DATA"].value_counts(dropna=False)
empty_polyline_count = (df["POLYLINE"] == "[]").sum()

display(missing_per_column)
display(missing_data_counts)
print(f"Empty POLYLINE rows: {empty_polyline_count:,}")

ORIGIN_CALL     39216
ORIGIN_STAND    25566
TRIP_ID             0
CALL_TYPE           0
TAXI_ID             0
TIMESTAMP           0
DAY_TYPE            0
MISSING_DATA        0
POLYLINE            0
dtype: int64

MISSING_DATA
False    50000
Name: count, dtype: int64

Empty POLYLINE rows: 200


## Parse `POLYLINE` va tao nhan ETA co ban

Moi dong trong `POLYLINE` la mot danh sach cac cap toa do `[longitude, latitude]`, lay mau moi 15 giay. Vi vay neu mot trajectory co `n` diem thi tong thoi gian chuyen di xap xi bang `(n - 1) * 15` giay. Day la cach don gian nhat de tao nhan ETA tu du lieu goc.

In [4]:
def parse_polyline(polyline_text: str):
    points = ast.literal_eval(polyline_text)
    return points if isinstance(points, list) else []


sample = df.copy()
sample["parsed_polyline"] = sample["POLYLINE"].apply(parse_polyline)
sample["num_points"] = sample["parsed_polyline"].apply(len)
sample["trip_duration_seconds"] = (sample["num_points"] - 1).clip(lower=0) * 15

preview_columns = [
    "TRIP_ID",
    "TIMESTAMP",
    "MISSING_DATA",
    "num_points",
    "trip_duration_seconds",
]

display(sample[preview_columns].head(10))
print("\nTrip duration summary (seconds):")
display(sample["trip_duration_seconds"].describe())

,TRIP_ID,TIMESTAMP,MISSING_DATA,num_points,trip_duration_seconds
0,1372636858620000589,1372636858,False,23,330
1,1372637303620000596,1372637303,False,19,270
2,1372636951620000320,1372636951,False,65,960
3,1372636854620000520,1372636854,False,43,630
4,1372637091620000337,1372637091,False,29,420
5,1372636965620000231,1372636965,False,26,375
6,1372637210620000456,1372637210,False,36,525
7,1372637299620000011,1372637299,False,34,495
8,1372637274620000403,1372637274,False,38,555
9,1372637905620000320,1372637905,False,19,270



Trip duration summary (seconds):


count    50000.000000
mean       702.593100
std        665.371018
min          0.000000
25%        405.000000
50%        600.000000
75%        855.000000
max      37725.000000
Name: trip_duration_seconds, dtype: float64